# GPT Mini Music Training on Kaggle (MIDI)

This notebook builds MIDI event tokens from a high-quality public dataset and trains your GPT-mini model using your project scripts.

Run cells in order.

In [ ]:
# Dependency check (no shell commands used)
try:
    import datasets  # noqa: F401
    print('datasets package is available')
except Exception as exc:
    raise ImportError(
        'The datasets package is required. Install it in the Kaggle environment before running.'
    ) from exc

In [ ]:
import os
import sys
from pathlib import Path
import torch

PROJECT_DIR = Path("/kaggle/working/transformer")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print('Project directory:', PROJECT_DIR)
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())

In [ ]:
# Verify required project files are present
PROJECT_DIR = Path('/kaggle/working/transformer')
required_files = [
    'train.py',
    'transformer.py',
    'decoder.py',
    'multi_head_attention.py',
    'feed_forward.py',
    'input_embedding.py',
    'layer_norm.py',
    'positional_encoding.py',
    'projection_layer.py',
    'residual_connection.py',
    'phase4_benchmark.py',
    'generate_tokens.py',
    'midi_audio.py',
    'midi_tokenizer.py',
]
missing = [f for f in required_files if not (PROJECT_DIR / f).exists()]
if missing:
    raise FileNotFoundError(f'Missing files in {PROJECT_DIR}: {missing}')
print('All required files found.')

## 1) Build MIDI Tokens from Dataset

In [ ]:
# Build MIDI token dataset (robust loader for MAESTRO-style datasets)
import json
import tempfile
from pathlib import Path

from datasets import load_dataset
from generate_tokens import SPECIAL_TOKENS, build_vocab, encode_tokens
from midi_tokenizer import midi_file_to_event_tokens

OUTPUT_DATA_DIR = '/kaggle/working/data'
output_dir = Path(OUTPUT_DATA_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

# Preferred high-quality piano MIDI corpus.
DATASET_CANDIDATES = [
    ('roszcz/maestro', None),
    ('roszcz/maestro', 'default'),
]
DATASET_SPLIT = 'train'
MAX_EXAMPLES = 2000
GRID_TICKS = 120
MAX_TIME_SHIFT = 64

def load_midi_dataset_with_fallback(candidates, split):
    last_error = None
    for name, config in candidates:
        try:
            if config is None:
                ds = load_dataset(name, split=split)
            else:
                ds = load_dataset(name, config, split=split)
            print(f'Loaded dataset: {name} config={config}')
            return ds, name, config
        except Exception as exc:
            print(f'Failed dataset={name} config={config}: {exc}')
            last_error = exc
    raise RuntimeError('Could not load any MIDI dataset candidate') from last_error

def extract_midi_bytes_or_path(example, tmp_dir: Path, index: int) -> Path:
    keys = ['midi', 'midi_bytes', 'midi_file', 'midi_filename', 'midi_path', 'path', 'file', 'filename']
    for key in keys:
        if key not in example:
            continue
        value = example[key]

        if isinstance(value, dict):
            for k in ('bytes', 'data'):
                maybe = value.get(k)
                if isinstance(maybe, (bytes, bytearray)):
                    out = tmp_dir / f'sample_{index:06d}.mid'
                    out.write_bytes(bytes(maybe))
                    return out
            for k in ('path', 'filename', 'name', 'file_name'):
                maybe = value.get(k)
                if isinstance(maybe, str) and maybe.lower().endswith(('.mid', '.midi')):
                    p = Path(maybe)
                    if p.exists():
                        return p

        if isinstance(value, (bytes, bytearray)):
            out = tmp_dir / f'sample_{index:06d}.mid'
            out.write_bytes(bytes(value))
            return out

        if isinstance(value, str) and value.lower().endswith(('.mid', '.midi')):
            p = Path(value)
            if p.exists():
                return p

    raise ValueError(f'No MIDI payload/path found. Keys: {list(example.keys())}')

dataset, used_dataset_name, used_dataset_config = load_midi_dataset_with_fallback(DATASET_CANDIDATES, DATASET_SPLIT)
print('Columns:', dataset.column_names)

tmp_dir = output_dir / 'midi_cache'
tmp_dir.mkdir(parents=True, exist_ok=True)

limit = min(MAX_EXAMPLES, len(dataset))
all_tokens = []
for i in range(limit):
    ex = dataset[i]
    midi_path = extract_midi_bytes_or_path(ex, tmp_dir, i)
    all_tokens.extend(
        midi_file_to_event_tokens(
            midi_path,
            grid_ticks=GRID_TICKS,
            max_time_shift=MAX_TIME_SHIFT,
        )
    )
    if (i + 1) % 50 == 0:
        print(f'Processed {i + 1}/{limit} MIDI files')

if not all_tokens:
    raise ValueError('No MIDI tokens were produced')

vocab_size_requested = max(2048, len(set(all_tokens)) + len(SPECIAL_TOKENS))
stoi = build_vocab(all_tokens, vocab_size=vocab_size_requested)
tokens_tensor = encode_tokens(all_tokens, stoi, add_bos_eos=False)

token_file = output_dir / 'midi_tokens.pt'
vocab_file = output_dir / 'midi_vocab.json'
stats_file = output_dir / 'midi_token_stats.json'

torch.save({'tokens': tokens_tensor}, token_file)

itos = [None] * len(stoi)
for tok, idx in stoi.items():
    itos[idx] = tok
vocab_payload = {'stoi': stoi, 'itos': itos, 'special_tokens': SPECIAL_TOKENS}
vocab_file.write_text(json.dumps(vocab_payload, ensure_ascii=True, indent=2), encoding='utf-8')

stats_payload = {
    'dataset_name': used_dataset_name,
    'dataset_config': used_dataset_config,
    'split': DATASET_SPLIT,
    'max_examples': MAX_EXAMPLES,
    'grid_ticks': GRID_TICKS,
    'max_time_shift': MAX_TIME_SHIFT,
    'num_tokens': int(tokens_tensor.numel()),
    'vocab_size_actual': int(len(stoi)),
    'vocab_size_requested': int(vocab_size_requested),
}
stats_file.write_text(json.dumps(stats_payload, ensure_ascii=True, indent=2), encoding='utf-8')

print('Saved token file:', token_file)
print('Saved vocab file:', vocab_file)
print('Saved stats file:', stats_file)
print('Token count:', tokens_tensor.numel())
print('Vocab size actual:', len(stoi))

In [ ]:
# Quick sanity check of MIDI token artifacts
tokens_path = Path('/kaggle/working/data/midi_tokens.pt')
vocab_path = Path('/kaggle/working/data/midi_vocab.json')
stats_path = Path('/kaggle/working/data/midi_token_stats.json')

print('tokens exists:', tokens_path.exists())
print('vocab exists:', vocab_path.exists())
print('stats exists:', stats_path.exists())

payload = torch.load(tokens_path, map_location='cpu')
tokens = payload['tokens'] if isinstance(payload, dict) else payload
print('token tensor shape:', tuple(tokens.shape))
print('token dtype:', tokens.dtype)

## 2) Train GPT Mini on MIDI Tokens

In [ ]:
# Auto-select DDP when 2 GPUs are available; otherwise run single-process training
import json
from train import TrainConfig, train, launch_ddp_training

vocab_payload = json.loads(Path('/kaggle/working/data/midi_vocab.json').read_text(encoding='utf-8'))
vocab_size = len(vocab_payload['stoi'])
print('Using vocab size:', vocab_size)

config = TrainConfig(
    token_file='/kaggle/working/data/midi_tokens.pt',
    output_dir='/kaggle/working/checkpoints',
    seq_len=512,
    batch_size=8,
    epochs=5,
    learning_rate=3e-4,
    weight_decay=0.1,
    warmup_steps=500,
    max_grad_norm=1.0,
    train_split=0.9,
    seed=42,
    vocab_size=vocab_size,
    d_model=512,
    num_heads=8,
    num_layers=8,
    d_ff=2048,
    max_seq_len=1024,
    dropout=0.1,
    num_workers=2,
    resume_from=None,
)

use_ddp = torch.cuda.is_available() and torch.cuda.device_count() >= 2
print('Using DDP:' if use_ddp else 'Using single-process training:', use_ddp)
if use_ddp:
    launch_ddp_training(config)
else:
    train(config)

In [ ]:
# The notebook now auto-launches DDP when 2 GPUs are available.
# No shell commands are needed for the default Kaggle notebook path.

## 3) Generate and Play Music from Trained Checkpoint

In [ ]:
import json
from IPython.display import Audio, display
from phase4_benchmark import build_model
from midi_tokenizer import PIECE_END, PIECE_START, decode_token_ids_to_tokens, render_token_ids_to_wav

ckpt_path = Path('/kaggle/working/checkpoints/best.pt')
vocab_path = Path('/kaggle/working/data/midi_vocab.json')
out_wav = Path('/kaggle/working/data/generated_music.wav')

if not ckpt_path.exists():
    raise FileNotFoundError(f'Checkpoint not found: {ckpt_path}')
if not vocab_path.exists():
    raise FileNotFoundError(f'Vocab file not found: {vocab_path}')

checkpoint = torch.load(ckpt_path, map_location='cpu')
cfg = checkpoint['config']
vocab_payload = json.loads(vocab_path.read_text(encoding='utf-8'))
stoi = vocab_payload['stoi']
itos = vocab_payload['itos']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model(
    vocab_size=cfg['vocab_size'],
    d_model=cfg['d_model'],
    num_heads=cfg['num_heads'],
    num_layers=cfg['num_layers'],
    d_ff=cfg['d_ff'],
    max_seq_len=cfg['max_seq_len'],
    dropout=cfg['dropout'],
    device=device,
 )
model.load_state_dict(checkpoint['model_state_dict'], strict=True)
model.eval()

start_id = stoi.get(PIECE_START)
end_id = stoi.get(PIECE_END)
if start_id is None:
    raise ValueError('PIECE_START token missing from vocab')

prompt = torch.tensor([[start_id]], dtype=torch.long, device=device)
with torch.no_grad():
    generated = model.generate(
        prompt,
        max_new_tokens=700,
        temperature=1.0,
        do_sample=True,
        top_k=50,
        eos_token_id=end_id,
        use_cache=True,
    )

generated_ids = generated[0].tolist()
generated_tokens = decode_token_ids_to_tokens(generated_ids, itos)
print('First tokens:', generated_tokens[:80])

render_token_ids_to_wav(generated_ids, itos, out_wav, step_seconds=0.12, step_ticks=100)
print('Generated WAV:', out_wav)
display(Audio(filename=str(out_wav), autoplay=False))

Training complete. Outputs are expected at:
- `/kaggle/working/checkpoints/best.pt`
- `/kaggle/working/data/midi_vocab.json`
- `/kaggle/working/data/generated_music.wav` (after running the generation cell)